## Backend Architecture

```text
Frontend
   |
   | HTTP Request
   v
Backend API
   |
   v
Face Search Engine
   |
   +----> MTCNN
   |
   +----> ArcFace
   |
   +----> SQLite Database
   |
   v
Ranked Candidates
   |
   v
JSON Response
   |
   v
Frontend

In [ ]:
from fastapi import FastAPI, UploadFile, File

print("FastAPI imported successfully!")

In [4]:
app = FastAPI(
    title="Missing Person Finder API",
    description="Backend API for face-based missing person search",
    version="1.0.0"
)

print("FastAPI application created!")

FastAPI application created!


In [5]:
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("CORS enabled for React frontend!")

CORS enabled for React frontend!


In [6]:
@app.get("/")
def home():

    return {
        "message": "Missing Person Finder API is running",
        "status": "OK"
    }


print("Home API endpoint created!")

Home API endpoint created!


In [7]:
@app.get("/health")
def health_check():

    return {
        "status": "healthy",
        "service": "Missing Person Finder API"
    }


print("Health-check endpoint created!")

Health-check endpoint created!


In [ ]:
import os  

import cv2

import numpy as np

import sqlite3

from deepface import DeepFace

from fastapi import FastAPI, UploadFile, File
from fastapi.testclient import TestClient
from fastapi.middleware.cors import CORSMiddleware

print("Face-search dependencies imported!")

Face-search dependencies imported!


In [3]:
DATABASE_PATH = "missing_persons.db"

connection = sqlite3.connect(
    DATABASE_PATH,
    check_same_thread=False
)

cursor = connection.cursor()

print("Database connection successful!")

Database connection successful!


In [8]:
def load_face_database():

    cursor.execute("""
    SELECT
        persons.person_id,
        persons.name,
        persons.age,
        persons.gender,
        persons.last_seen_location,
        persons.contact_information,
        persons.report_date,
        persons.status,
        photos.photo_id,
        photos.file_path,
        embeddings.embedding,
        embeddings.model_name
    FROM embeddings
    JOIN photos
        ON embeddings.photo_id = photos.photo_id
    JOIN persons
        ON photos.person_id = persons.person_id
    WHERE persons.status = 'MISSING'
    """)

    records = cursor.fetchall()

    database = []

    for record in records:

        embedding = np.frombuffer(
            record[10],
            dtype=np.float64
        )

        database.append({
            "person_id": record[0],
            "name": record[1],
            "age": record[2],
            "gender": record[3],
            "last_seen_location": record[4],
            "contact_information": record[5],
            "report_date": record[6],
            "status": record[7],
            "photo_id": record[8],
            "file_path": record[9],
            "embedding": embedding,
            "model_name": record[11]
        })

    return database


face_database = load_face_database()

print("Face database loaded!")
print("Active embeddings:", len(face_database))

Face database loaded!
Active embeddings: 3


In [9]:
def detect_faces(image_path):

    detections = DeepFace.extract_faces(
        img_path=image_path,
        detector_backend="mtcnn",
        enforce_detection=False,
        align=True
    )

    valid_faces = [
        detection
        for detection in detections
        if detection["confidence"] > 0
    ]

    return valid_faces


print("Face detection function ready!")

Face detection function ready!


In [10]:
def generate_face_embeddings(image_path):

    faces = detect_faces(image_path)

    embeddings = []

    for face in faces:

        face_crop = face["face"]

        embedding = DeepFace.represent(
            img_path=face_crop,
            model_name="ArcFace",
            detector_backend="skip",
            enforce_detection=False
        )[0]["embedding"]

        embeddings.append(
            np.array(embedding)
        )

    return embeddings


print("Face embedding function ready!")

Face embedding function ready!


In [11]:
def cosine_distance(embedding_a, embedding_b):

    similarity = np.dot(
        embedding_a,
        embedding_b
    ) / (
        np.linalg.norm(embedding_a)
        * np.linalg.norm(embedding_b)
    )

    distance = 1 - similarity

    return distance


print("Cosine distance function ready!")

Cosine distance function ready!


In [12]:
def calculate_person_statistics(query_embedding):

    face_database = load_face_database()

    person_data = {}

    for record in face_database:

        distance = cosine_distance(
            query_embedding,
            record["embedding"]
        )

        person_id = record["person_id"]

        if person_id not in person_data:

            person_data[person_id] = {
                "person_id": person_id,
                "name": record["name"],
                "age": record["age"],
                "gender": record["gender"],
                "last_seen_location": record["last_seen_location"],
                "contact_information": record["contact_information"],
                "report_date": record["report_date"],
                "status": record["status"],
                "distances": [],
                "best_distance": distance,
                "best_photo_id": record["photo_id"]
            }

        person_data[person_id]["distances"].append(distance)

        # Update the closest reference photo
        if distance < person_data[person_id]["best_distance"]:
            person_data[person_id]["best_distance"] = distance
            person_data[person_id]["best_photo_id"] = record["photo_id"]

    results = []

    for person in person_data.values():

        distances = person["distances"]

        results.append({
            "person_id": person["person_id"],
            "name": person["name"],
            "age": person["age"],
            "gender": person["gender"],
            "last_seen_location": person["last_seen_location"],
            "contact_information": person["contact_information"],
            "report_date": person["report_date"],
            "status": person["status"],
            "best_distance": person["best_distance"],
            "average_distance": np.mean(distances),
            "best_photo_id": person["best_photo_id"],
            "reference_photos": len(distances)
        })

    results.sort(
        key=lambda x: x["best_distance"]
    )

    return results


print("Person statistics function updated!")

Person statistics function updated!


In [ ]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg"
)

formatted_result = format_search_results(
    result
)

print(formatted_result)

In [14]:
def search_missing_person(image_path, top_k=5, threshold=0.68):

    # Generate query embedding
    query_embeddings = generate_face_embeddings(image_path)

    # No face
    if len(query_embeddings) == 0:
        return {
            "status": "NO_FACE",
            "results": []
        }

    # Multiple faces
    if len(query_embeddings) > 1:
        return {
            "status": "MULTIPLE_FACES",
            "results": []
        }

    # Single face
    query_embedding = query_embeddings[0]

    # Calculate person-level statistics
    results = calculate_person_statistics(
        query_embedding
    )

    # Empty database
    if len(results) == 0:
        return {
            "status": "NO_DATABASE_RECORDS",
            "results": []
        }

    # Keep top candidates
    results = results[:top_k]

    # Apply threshold
    if results[0]["best_distance"] < threshold:
        status = "POTENTIAL_MATCH"
    else:
        status = "NO_RELIABLE_MATCH"

    return {
        "status": status,
        "results": results
    }


print("Backend search function ready!")

Backend search function ready!


In [15]:
def format_search_results(result):

    formatted = {
        "status": result["status"],
        "candidates": []
    }

    for rank, person in enumerate(result["results"], start=1):

        candidate = {
            "rank": rank,

            "person": {
                "person_id": int(person["person_id"]),
                "name": person["name"],
                "age": int(person["age"]) if person["age"] is not None else None,
                "gender": person["gender"],
                "last_seen_location": person["last_seen_location"],
                "contact_information": person["contact_information"],
                "report_date": person["report_date"],
                "status": person["status"]
            },

            "match": {
                "best_distance": float(person["best_distance"]),
                "average_distance": float(person["average_distance"]),
                "best_photo_id": int(person["best_photo_id"]),
                "reference_photos": int(person["reference_photos"])
            }
        }

        formatted["candidates"].append(candidate)

    return formatted


print("API response formatter ready!")

API response formatter ready!


In [16]:
@app.post("/search")
async def search_face(
    file: UploadFile = File(...)
):

    # Read uploaded file
    image_bytes = await file.read()

    # Save temporary image
    temp_path = "temp_query.jpg"

    with open(temp_path, "wb") as image_file:
        image_file.write(image_bytes)

    # Run face search
    result = search_missing_person(
        temp_path
    )

    # Convert result to API-friendly format
    formatted_result = format_search_results(
        result
    )

    # Remove temporary file
    if os.path.exists(temp_path):
        os.remove(temp_path)

    return formatted_result


print("POST /search endpoint created!")

POST /search endpoint created!


In [17]:
from fastapi.responses import FileResponse


@app.get("/photo/{photo_id}")
def get_photo(photo_id: int):

    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT file_path
        FROM photos
        WHERE photo_id = ?
        """,
        (photo_id,)
    )

    row = cursor.fetchone()

    if row is None:
        return {
            "error": "Photo not found"
        }

    photo_path = row[0]

    if not os.path.exists(photo_path):
        return {
            "error": "Photo file does not exist"
        }

    return FileResponse(photo_path)


print("GET /photo/{photo_id} endpoint created!")

GET /photo/{photo_id} endpoint created!


In [ ]:
client = TestClient(app)

print("API test client created!")

In [ ]:
response = client.post(
    "/search",
    files={
        "file": (
            "img4.jpg",
            open("deepface_repo/tests/unit/dataset/img4.jpg", "rb"),
            "image/jpeg"
        )
    }
)

print("Status code:", response.status_code)
print(response.json())

In [ ]:
response = client.get("/health")

print("Status code:", response.status_code)
print(response.json())

In [ ]:
response = client.post(
    "/search",
    files={
        "file": (
            "test_no_face.jpg",
            open("test_no_face.jpg", "rb"),
            "image/jpeg"
        )
    }
)

print("Status code:", response.status_code)
print(response.json())

In [ ]:
response = client.post(
    "/search",
    files={
        "file": (
            "test_multiple_faces.jpg",
            open("test_multiple_faces.jpg", "rb"),
            "image/jpeg"
        )
    }
)

print("Status code:", response.status_code)
print(response.json())

In [ ]:
response = client.post(
    "/search",
    files={
        "file": (
            "img13.jpg",
            open(
                "deepface_repo/tests/unit/dataset/img13.jpg",
                "rb"
            ),
            "image/jpeg"
        )
    }
)

print("Status code:", response.status_code)
print(response.json())

In [ ]:
import threading
import uvicorn

def start_backend():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000
    )

backend_thread = threading.Thread(
    target=start_backend,
    daemon=True
)

backend_thread.start()

print("FastAPI server started on http://127.0.0.1:8000")

FastAPI server started on http://127.0.0.1:8000


INFO:     Started server process [833]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:58932 - "GET /photo/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58944 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:58951 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:58977 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:59514 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:59519 - "GET /photo/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:59521 - "GET /photo/2 HTTP/1.1" 200 OK
INFO:     127.0.0.1:59903 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:60068 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:60069 - "GET /photo/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:60118 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:60151 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:60151 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:60226 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:60291 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:62396 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:62689 - "POST /search HTTP/1.1" 200 

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/sadiquenomani/traceback-phase1/venv/lib/python3.12/site-packages/uvicorn/protocols/http/h11_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sadiquenomani/traceback-phase1/venv/lib/python3.12/site-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sadiquenomani/traceback-phase1/venv/lib/python3.12/site-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/sadiquenomani/traceback-phase1/venv/lib/python3.12/site-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/sadiquenomani/traceback-phase1/venv/lib/python3.12/site

INFO:     127.0.0.1:49763 - "GET /photo/11 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49764 - "GET /photo/24 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49771 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:49772 - "GET /photo/26 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49773 - "GET /photo/11 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49800 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:49803 - "GET /photo/9 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49801 - "GET /photo/25 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49802 - "GET /photo/17 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49841 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:49844 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:49845 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:49847 - "GET /photo/28 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49848 - "GET /photo/16 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49846 - "GET /photo/8 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49855 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:49856 - "GET /photo/28 HTTP/

## API Endpoints

### GET `/`

Checks whether the Missing Person Finder API is running.

### GET `/health`

Returns the health status of the backend service.

### POST `/search`

Accepts an uploaded image and performs:

1. Face detection using MTCNN
2. Face embedding generation using ArcFace
3. Search against registered missing-person embeddings
4. Person-level candidate ranking
5. Threshold-based match classification
6. JSON response generation

### Search Statuses

- `POTENTIAL_MATCH` — a candidate is within the configured threshold.
- `NO_RELIABLE_MATCH` — candidates exist, but none is within the threshold.
- `NO_FACE` — no valid face was detected.
- `MULTIPLE_FACES` — more than one face was detected.
- `NO_DATABASE_RECORDS` — no active missing-person records are available.

The API is designed to return potential candidates for **human verification**, rather than making an automatic identity decision.

## Conclusion

The backend API for the Missing Person Finder prototype was successfully implemented using FastAPI.

The API provides:

- A root endpoint for service information.
- A health-check endpoint.
- An image-upload search endpoint.
- MTCNN-based face detection.
- ArcFace-based 512-dimensional face embeddings.
- SQLite-based missing-person search.
- Person-level candidate ranking.
- Threshold-based match classification.
- JSON-compatible responses.

The `/search` endpoint was tested with four important scenarios:

- A valid single-face image returned `POTENTIAL_MATCH`.
- An image without a face returned `NO_FACE`.
- An image containing multiple faces returned `MULTIPLE_FACES`.
- An unknown person returned `NO_RELIABLE_MATCH`.

These tests confirm that the backend search pipeline is functioning correctly for the selected prototype cases.

The current API is still a development prototype. A production implementation would require stronger input validation, authentication and authorization, secure handling of personal information, improved database management, concurrent request handling, persistent image storage, logging, and deployment configuration.

The next stage is to connect the backend API with a frontend interface so users can upload photographs and view ranked potential matches.

In [20]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT photo_id, file_path
    FROM photos
    WHERE photo_id = ?
    """,
    (3,)
)

row = cursor.fetchone()

print("Database row:", row)

if row:
    print("Path:", row[1])
    print("Exists:", os.path.exists(row[1]))
    print("Absolute path:", os.path.abspath(row[1]))

Database row: (3, 'deepface_repo/tests/unit/dataset/img2.jpg')
Path: deepface_repo/tests/unit/dataset/img2.jpg
Exists: True
Absolute path: /Users/sadiquenomani/traceback-phase1/deepface_repo/tests/unit/dataset/img2.jpg
